In [15]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [16]:
class Value:

    def __init__(self, data, _children=(), _op="", label=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"
    
    # ── Primitives (depend on nothing else in this class) ──────────────────
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other) 
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data *  other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other): # self ** other
        assert isinstance(other, (int, float)), "Only supporting int/float powers for now!"
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += other * (self.data**(other-1)) * out.grad
        out._backward = _backward
        return out
    
    def exp(self):
        x = self.data
        out = Value(math.exp(x),(self,),"exp")

        def _backward():
            self.grad +=  out.data * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x)-1)/(math.exp(2*x)+1)
        out = Value(t, (self, ), "tanh")

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    # ── Level 1: depend only on primitives above ───────────────────────────
    
    def __neg__(self): # -self # calls self.__mul__(-1)
        return self * -1

    def __radd__(self, other): # other + self; calls other.__add__(self)
        return self + other
        
    def __rmul__(self, other): # other * self; calls other.__mul__(self)
        return self * other

    # ── Level 2: depend on Level 1 and/or primitives ──────────────────────
    
    def __sub__(self, other): # self - other; uses __add__(), __neg__()
        return self + (-other)
    
    def __truediv__(self, other): # self / other; uses __mul__(), __pow__()
        return self * (other**-1)
    
    # ── Backward pass ─────────────────────────────────────────────────────
    
    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
                visited.add(v)
        
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

In [17]:
# Topological sort



In [33]:
class Neuron:
  def __init__(self, nin):
    self.w = [Value(random.uniform(-1, +1)) for _ in range(nin)]
    self.b = Value(random.uniform(-1, +1))
  
  def __call__(self, x):
    act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
    out = act.tanh()
    return out

# x = [2.0, 3.0]
# n = Neuron(2)
# n(x)

class Layer:
  def __init__(self, nin, nout):
    self.neurons = [Neuron(nin) for _ in range(nout)]
  
  def __call__(self, x):
    outs = [n(x) for n in self.neurons]
    return outs
  
x = [2.0, 3.0]
L = Layer(2, 3)
L(x)

[Value(data=-0.9836036165015211),
 Value(data=-0.9339074379731268),
 Value(data=0.9545647744379046)]